# Survey Samples and Conflict Exposure

**Purpose:** Prepare and validate post-war survey samples and construct conflict-exposure measures for Liberia and Sierra Leone.

**Date started:** 2026-10-12 (earliest recorded Git version).  
**Status:** Work in progress.  
**Code style:** Use descriptive `snake_case` names, group imports, and keep code cells focused.

In [1]:
# Importing packages (each one once)
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import requests
from matplotlib import animation
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from IPython.display import Video, Image, display
from ipumspy import readers

In [2]:
# Locate the repository from the notebook working directory.
project_dir = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(project_dir, ".aea")):
    parent_dir = os.path.dirname(project_dir)
    if parent_dir == project_dir:
        raise FileNotFoundError("Could not locate the HHEI project root.")
    project_dir = parent_dir

data_dir = os.path.join(project_dir, "data")
ipums_data_dir = os.path.join(data_dir, "ipums_dhs", "raw")
dhs_processed_dir = os.path.join(data_dir, "dhs", "processed")
ucdp_raw_dir = os.path.join(data_dir, "ucdp", "raw")
ucdp_derived_dir = os.path.join(data_dir, "ucdp", "derived")
geography_dir = os.path.join(data_dir, "geography", "raw")
output_dir = os.path.join(project_dir, "outputs")

for directory in (dhs_processed_dir, ucdp_derived_dir, geography_dir, output_dir):
    os.makedirs(directory, exist_ok=True)

wr_ddi_path = os.path.join(ipums_data_dir, "dhs_women_lbr_sl_ddi.xml")
mr_ddi_path = os.path.join(ipums_data_dir, "dhs_men_lbr_sl_ddi.xml")
wr_data_path = os.path.join(ipums_data_dir, "dhs_women_lbr_sl.dat.gz")
mr_data_path = os.path.join(ipums_data_dir, "dhs_men_lbr_sl.dat.gz")

## 1. Load the data

The DDI (`.xml`) is the codebook: variable names, descriptions and value labels. The `.dat.gz` file holds the numbers. `read_microdata` gives you plain numbers, and the labels stay in the DDI object, so we keep both around.

In [3]:
ddi_wr_obj = readers.read_ipums_ddi(wr_ddi_path)
ddi_mr_obj = readers.read_ipums_ddi(mr_ddi_path)

wr_df = readers.read_microdata(ddi_wr_obj, wr_data_path)   # women's recode
mr_df = readers.read_microdata(ddi_mr_obj, mr_data_path)   # men's recode

print("Women's file:", wr_df.shape)
print("Men's file:  ", mr_df.shape)

/opt/homebrew/lib/python3.14/site-packages/ipumspy/readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


Women's file: (40363, 88)
Men's file:   (20669, 66)


## 2. Helpers for reading value labels

These three small functions are used throughout. They look codes up **by their label** (e.g. "Always") instead of hard-coding numbers like 95, so the code stays correct if IPUMS codes differ between files.

* `labels_for(ddi, var)` returns `{code: label}` for one variable (an empty dict if the variable is not in that file).
* `codes_matching(ddi, var, pattern)` returns the set of codes whose label matches a regular expression (case-insensitive).
* `SPECIAL_CODES` is the pattern for "no real answer" labels: missing, NIU (not in universe), don't know, and so on.

In [4]:
def labels_for(ddi, var):
    """{code: label} for one variable, or {} if the variable isn't in this DDI."""
    for v in ddi.data_description:
        if v.name == var:
            return {int(code): str(label) for label, code in v.codes.items()}
    return {}


def codes_matching(ddi, var, pattern):
    """Set of codes whose label matches `pattern` (a case-insensitive regex)."""
    return {code for code, label in labels_for(ddi, var).items()
            if re.search(pattern, label, flags=re.IGNORECASE)}


SPECIAL_CODES = r"missing|\bniu\b|not in universe|don.?t know|refused|no response|inconsistent"

# Quick example: what does LIT2 mean?
labels_for(ddi_wr_obj, "LIT2")

{0: 'Cannot read at all',
 10: 'Reads',
 11: 'Able to read only part of sentence',
 12: 'Able to read whole sentence',
 20: 'Not ascertained (blind, diff. language)',
 21: 'No card with required language',
 22: 'Blind or visually impaired',
 98: 'Missing'}

## 3. Reference tables

Two lookup tables you can filter whenever you need to know what a variable or code means:

* `variables`: one row per variable, with its label and which file it comes from.
* `all_codes`: one row per variable-code pair. Filter it, e.g. `all_codes[all_codes["variable"] == "LIT2"]`.

In [5]:
variables = pd.DataFrame([
    {"file": file, "variable": v.name, "label": v.label}
    for file, ddi in [("women", ddi_wr_obj), ("men", ddi_mr_obj)]
    for v in ddi.data_description
])

all_codes = pd.DataFrame([
    {"file": file, "variable": v.name, "code": code, "label": label}
    for file, ddi in [("women", ddi_wr_obj), ("men", ddi_mr_obj)]
    for v in ddi.data_description
    for label, code in v.codes.items()
])

all_codes[all_codes["variable"].isin(["LIT2", "LIT2MN"])]

,file,variable,code,label
1710,women,LIT2,0,Cannot read at all
1711,women,LIT2,10,Reads
1712,women,LIT2,11,Able to read only part of sentence
1713,women,LIT2,12,Able to read whole sentence
1714,women,LIT2,20,"Not ascertained (blind, diff. language)"
1715,women,LIT2,21,No card with required language
1716,women,LIT2,22,Blind or visually impaired
1717,women,LIT2,98,Missing
3626,men,LIT2MN,0,Cannot read at all
3627,men,LIT2MN,10,Reads


## 4. Stack the men's and women's files

The two files are separate samples of different people, so we **stack** them (append rows) rather than merging. Many men's variables have an `MN` suffix, so `variable_pairs` gives each concept one shared name.

**Fix vs. the earlier version:** previously, a misspelled or missing men's variable was skipped silently and men just got `NaN`. Now the cell prints any variable it can't find, so you notice. Residence and the person weight were also added to the pairs.

In [6]:
# harmonized name: (women's variable, men's variable)
variable_pairs = {
    "AGE": ("AGE", "AGEMN"),
    "WEIGHT": ("PERWEIGHT", "PERWEIGHTMN"),
    "RESIDENCE_YEARS": ("RESIDEINTYR", "RESIDEINTYRMN"),
    "EDUCATION_LEVEL": ("EDUCLVL", "EDUCLVLMN"),
    "EDUCATION_YEARS": ("EDYRTOTAL", "EDYRTOTALMN"),
    "LITERACY": ("LIT2", "LIT2MN"),
    "LITERACY_BRIDGE": ("LITBRIG", "LITBRIGMN"),
    "LITERACY_PROGRAM_EVER": ("LITPROGRAMEV", "LITPROGRAMEVMN"),
    "WEALTH_QUINTILE": ("WEALTHQ", "WEALTHQMN"),
    "URBAN_RURAL": ("URBAN", "URBANMN"),
    "WAR_STAYED_HOME": ("MIGRWARSTAY", "MIGRWARSTAYMN"),
    "WAR_RETURNED": ("MIGRWARREL", "MIGRWARRELMN"),
    "WAR_DISPLACED_CAMP": ("MIGRWARCAMP", "MIGRWARCAMPMN"),
    "WAR_DISPLACED_BUSH": ("MIGRWARBUSH", "MIGRWARBUSHMN"),
    "WAR_DISPLACED_ABROAD": ("MIGRWARABROAD", "MIGRWARABROADMN"),
    "WAR_DISPLACED_OTHER": ("MIGRWAROTH", "MIGRWAROTHMN"),
}

# CHECK: every source variable should exist in its file
missing_women = [w for w, m in variable_pairs.values() if w not in wr_df.columns]
missing_men = [m for w, m in variable_pairs.values() if m not in mr_df.columns]
print("Not found in women's file:", missing_women or "none")
print("Not found in men's file:  ", missing_men or "none")


def prepare_recode(data, recode, sex, source_index):
    """Copy each source variable into its harmonized name. Original columns are kept.
    A variable missing from this file becomes an empty column (and shows up in the check above)."""
    prepared = data.copy()
    prepared["RECODE"] = recode
    prepared["SEX"] = sex
    for harmonized_name, source_names in variable_pairs.items():
        source_name = source_names[source_index]
        if source_name in prepared.columns:
            prepared[harmonized_name] = prepared[source_name]
        else:
            prepared[harmonized_name] = pd.Series(pd.NA, index=prepared.index, dtype="Float64")
    return prepared


combined_df = pd.concat(
    [prepare_recode(wr_df, "WR", "Female", 0),
     prepare_recode(mr_df, "MR", "Male", 1)],
    ignore_index=True,
    sort=False,
)

# Country names come from the DDI labels rather than a hand-typed list
country_labels = {**labels_for(ddi_mr_obj, "COUNTRY"), **labels_for(ddi_wr_obj, "COUNTRY")}
combined_df["COUNTRY_NAME"] = combined_df["COUNTRY"].map(country_labels)
print("Combined records:", combined_df.shape)
combined_df["COUNTRY_NAME"].value_counts(dropna=False)

Not found in women's file: none
Not found in men's file:   none
Combined records: (61032, 148)


COUNTRY_NAME
Sierra Leone    34574
Liberia         26458
Name: count, dtype: int64

## 5. Check: which surveys and variables do we actually have?

The extract includes Liberia 2007 and 2013, plus Sierra Leone 2008 and 2013. The 2013 waves have no residence-years responses, so they appear in the wave-specific exports but are excluded from the strict lifelong-resident analysis sample.

The second table shows the share of non-missing values for each harmonized variable by country, year and sex. A `0.0` means that variable doesn't exist for that group, such as the residence and war-migration questions in some waves.

In [7]:
display(combined_df.groupby(["COUNTRY_NAME", "YEAR", "SEX"]).size().rename("records"))

harmonized = list(variable_pairs)
availability = (
    combined_df[harmonized].notna()
    .groupby([combined_df["COUNTRY_NAME"], combined_df["YEAR"], combined_df["SEX"]])
    .mean()
    .round(2)
    .T            # transpose: variables as rows, groups as columns
)
availability

COUNTRY_NAME  YEAR  SEX   
Liberia       2007  Female     7092
                    Male       6009
              2013  Female     9239
                    Male       4118
Sierra Leone  2008  Female     7374
                    Male       3280
              2013  Female    16658
                    Male       7262
Name: records, dtype: int64

COUNTRY_NAME          Liberia                  Sierra Leone                 
YEAR                     2007        2013              2008        2013     
SEX                    Female Male Female Male       Female Male Female Male
AGE                       1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
WEIGHT                    1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
RESIDENCE_YEARS           1.0  1.0    0.0  0.0          1.0  1.0    0.0  0.0
EDUCATION_LEVEL           1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
EDUCATION_YEARS           1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
LITERACY                  1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
LITERACY_BRIDGE           1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
LITERACY_PROGRAM_EVER     1.0  1.0    0.0  0.0          1.0  1.0    0.0  0.0
WEALTH_QUINTILE           1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
URBAN_RURAL               1.0  1.0    1.0  1.0          1.0  1.0    1.0  1.0
WAR_STAYED_HOME           1.0  1.0    0.0  0.0          0.0  0.0    0.0  0.0
WAR_RETURNED              1.0  1.0    0.0  0.0          0.0  0.0    0.0  0.0
WAR_DISPLACED_CAMP        1.0  1.0    0.0  0.0          0.0  0.0    0.0  0.0
WAR_DISPLACED_BUSH        1.0  1.0    0.0  0.0          0.0  0.0    0.0  0.0
WAR_DISPLACED_ABROAD      1.0  1.0    0.0  0.0          0.0  0.0    0.0  0.0
WAR_DISPLACED_OTHER       1.0  1.0    0.0  0.0          0.0  0.0    0.0  0.0

## 6. Respondent geography

`ADMIN_AREA` uses the comparable integrated geography: groups of counties in Liberia and groups of districts in Sierra Leone. The 2013 waves also provide finer county/district geography, shown in `ADMIN_UNIT_NAME` with its level in `ADMIN_UNIT_LEVEL`; Liberia 2007 and Sierra Leone 2008 remain at the broader group level. These variables describe where respondents were interviewed, not necessarily where they lived during the war.

`GPS_LATITUDE` and `GPS_LONGITUDE` are standardized WGS84 coordinates for DHS primary-sampling-unit (cluster) centers, not individuals' exact homes. DHS randomly displaces these coordinates for privacy: up to 2 km in urban areas, up to 5 km for 99% of rural clusters, and up to 10 km for 1% of rural clusters.

The table below uses one coordinate pair per `DHSID`, separately by country, year, and sex. It assigns each cluster to one exclusive band based on its nearest other distinct cluster center: 0-2 km, over 2-5 km, over 5-10 km, or over 10 km. The people columns count respondents in clusters in each band, using only respondents with GPS coordinates; their shares use all GPS-located respondents in that country-year-sex group as the denominator. The distance bands do not overlap, and their shares sum to 1. These are distances between displaced centers, not recovered distances between the true PSU locations.

In [8]:
is_liberia = combined_df["COUNTRY_NAME"].eq("Liberia")
is_sierra_leone = combined_df["COUNTRY_NAME"].eq("Sierra Leone")

combined_df["ADMIN_AREA"] = combined_df["GEO_LR2007_2019"].where(is_liberia, combined_df["GEO_SL2008_2019"])

lr_region_labels = {**labels_for(ddi_mr_obj, "GEO_LR2007_2019"), **labels_for(ddi_wr_obj, "GEO_LR2007_2019")}
sl_region_labels = {**labels_for(ddi_mr_obj, "GEO_SL2008_2019"), **labels_for(ddi_wr_obj, "GEO_SL2008_2019")}
combined_df["ADMIN_AREA_NAME"] = combined_df["ADMIN_AREA"].map(lr_region_labels).where(
    is_liberia, combined_df["ADMIN_AREA"].map(sl_region_labels)
)

lr_county_labels = {**labels_for(ddi_mr_obj, "GEO_LR2013_2019"), **labels_for(ddi_wr_obj, "GEO_LR2013_2019")}
sl_district_labels = {**labels_for(ddi_mr_obj, "GEOALT_SL2013_2019"), **labels_for(ddi_wr_obj, "GEOALT_SL2013_2019")}
lr_county_name = combined_df["GEO_LR2013_2019"].map(lr_county_labels).where(is_liberia)
sl_district_name = combined_df["GEOALT_SL2013_2019"].map(sl_district_labels).where(is_sierra_leone)
detailed_admin_name = lr_county_name.combine_first(sl_district_name)
combined_df["ADMIN_UNIT_NAME"] = detailed_admin_name.combine_first(combined_df["ADMIN_AREA_NAME"])
combined_df["ADMIN_UNIT_LEVEL"] = np.select(
    [is_liberia & detailed_admin_name.notna(), is_sierra_leone & detailed_admin_name.notna(), is_liberia],
    ["County", "District", "County group"],
    default="District group",
)

combined_df["GPS_LATITUDE"] = pd.to_numeric(combined_df["GPSLAT"], errors="coerce").mask(
    pd.to_numeric(combined_df["GPSLAT"], errors="coerce").eq(0)
)
combined_df["GPS_LONGITUDE"] = pd.to_numeric(combined_df["GPSLONG"], errors="coerce").mask(
    pd.to_numeric(combined_df["GPSLONG"], errors="coerce").eq(0)
)

# CHECK: standardized administrative detail and GPS coverage by wave
combined_df.groupby(["COUNTRY_NAME", "YEAR"]).agg(
    respondents=("COUNTRY", "size"),
    admin_detail=("ADMIN_UNIT_NAME", "nunique"),
    gps_coordinates=("GPS_LATITUDE", "count"),
)

respondents  admin_detail  gps_coordinates
COUNTRY_NAME YEAR                                            
Liberia      2007        13101             5            12840
             2013        13357            15            13357
Sierra Leone 2008        10654             4            10571
             2013        23920            13            23920

In [23]:
wave_order = [
    ("Liberia", 2007),
    ("Liberia", 2013),
    ("Sierra Leone", 2008),
    ("Sierra Leone", 2013),
]
distance_bands = ["0-2 km", ">2-5 km", ">5-10 km", ">10 km"]
cluster_spacing_rows = []

for country, year in wave_order:
    for sex in ["Female", "Male"]:
        wave_people = combined_df.loc[
            combined_df["COUNTRY_NAME"].eq(country)
            & combined_df["YEAR"].eq(year)
            & combined_df["SEX"].eq(sex),
            ["DHSID", "GPS_LATITUDE", "GPS_LONGITUDE"],
        ].dropna()
        coordinates = wave_people.drop_duplicates(subset="DHSID")
        people_by_cluster = (
            wave_people.groupby("DHSID").size().reindex(coordinates["DHSID"]).to_numpy()
        )

        nearest_distance_km = np.array([])
        if len(coordinates) == 1:
            nearest_distance_km = np.array([np.inf])
        elif len(coordinates) > 1:
            locations = gpd.GeoDataFrame(
                coordinates,
                geometry=gpd.points_from_xy(coordinates["GPS_LONGITUDE"], coordinates["GPS_LATITUDE"]),
                crs="EPSG:4326",
            )
            locations = locations.to_crs(locations.estimate_utm_crs())
            xy_km = np.column_stack([locations.geometry.x, locations.geometry.y]) / 1000
            pairwise_km = np.sqrt(((xy_km[:, None, :] - xy_km[None, :, :]) ** 2).sum(axis=2))
            np.fill_diagonal(pairwise_km, np.inf)
            nearest_distance_km = pairwise_km.min(axis=1)

        band_masks = [
            nearest_distance_km <= 2,
            (nearest_distance_km > 2) & (nearest_distance_km <= 5),
            (nearest_distance_km > 5) & (nearest_distance_km <= 10),
            nearest_distance_km > 10,
        ]
        row = {
            "Country": country,
            "Year": year,
            "Sex": sex,
            "GPS clusters": len(coordinates),
            "GPS-located people": len(wave_people),
        }
        for band, in_band in zip(distance_bands, band_masks):
            cluster_count = int(in_band.sum())
            people_count = int(people_by_cluster[in_band].sum())
            row[f"Clusters {band}"] = cluster_count
            row[f"Share of clusters {band}"] = (
                cluster_count / len(coordinates) if len(coordinates) else np.nan
            )
            row[f"People in clusters {band}"] = people_count
            row[f"Share of people {band}"] = (
                people_count / len(wave_people) if len(wave_people) else np.nan
            )
        cluster_spacing_rows.append(row)

cluster_spacing_table = pd.DataFrame(cluster_spacing_rows).set_index(["Country", "Year", "Sex"])
cluster_spacing_table

GPS clusters  GPS-located people  Clusters 0-2 km  \
Country      Year Sex                                                         
Liberia      2007 Female           291                6951              108   
                  Male             291                5889              108   
             2013 Female           322                9239               69   
                  Male             322                4118               69   
Sierra Leone 2008 Female           350                7320              118   
                  Male             350                3251              118   
             2013 Female           435               16658              123   
                  Male             435                7262              123   

                          Share of clusters 0-2 km  People in clusters 0-2 km  \
Country      Year Sex                                                           
Liberia      2007 Female                  0.371134                       3044   
                  Male                    0.371134                       2419   
             2013 Female                  0.214286                       2253   
                  Male                    0.214286                        972   
Sierra Leone 2008 Female                  0.337143                       2612   
                  Male                    0.337143                       1145   
             2013 Female                  0.282759                       5536   
                  Male                    0.282759                       2427   

                          Share of people 0-2 km  Clusters >2-5 km  \
Country      Year Sex                                                
Liberia      2007 Female                0.437923                35   
                  Male                  0.410766                35   
             2013 Female                0.243858                59   
                  Male                  0.236037                59   
Sierra Leone 2008 Female                0.356831                71   
                  Male                  0.352199                71   
             2013 Female                0.332333                87   
                  Male                  0.334205                87   

                          Share of clusters >2-5 km  \
Country      Year Sex                                 
Liberia      2007 Female                   0.120275   
                  Male                     0.120275   
             2013 Female                   0.183230   
                  Male                     0.183230   
Sierra Leone 2008 Female                   0.202857   
                  Male                     0.202857   
             2013 Female                   0.200000   
                  Male                     0.200000   

                          People in clusters >2-5 km  Share of people >2-5 km  \
Country      Year Sex                                                           
Liberia      2007 Female                         809                 0.116386   
                  Male                           741                 0.125828   
             2013 Female                        1669                 0.180647   
                  Male                           705                 0.171200   
Sierra Leone 2008 Female                        1450                 0.198087   
                  Male                           622                 0.191326   
             2013 Female                        3225                 0.193601   
                  Male                          1389                 0.191270   

                          Clusters >5-10 km  Share of clusters >5-10 km  \
Country      Year Sex                                                     
Liberia      2007 Female                 47                    0.161512   
                  Male                   47                    0.161512   
             2013 Female                 85                  

## 7. Turn special codes into missing values

Codes like 98 ("Missing") or 8 ("NIU") are not real values. Left in, a 98 counts as 98 years of schooling. Here they become `NaN`, using each file's own labels.

Only the **harmonized** copies are changed; the original IPUMS columns stay untouched so you can always compare.

Two variables are deliberately *not* in this list:
* `RESIDENCE_YEARS`: code "Always" is a real answer (the person never moved), and it defines our sample. It is handled in step 8.
* `LITERACY`: it has several categories that each need a decision, handled in step 9.

In [10]:
vars_to_clean = [
    "EDUCATION_LEVEL", "EDUCATION_YEARS", "LITERACY_BRIDGE", "LITERACY_PROGRAM_EVER",
    "WAR_STAYED_HOME", "WAR_RETURNED", "WAR_DISPLACED_CAMP",
    "WAR_DISPLACED_BUSH", "WAR_DISPLACED_ABROAD", "WAR_DISPLACED_OTHER",
]
ddi_by_sex = {"Female": (ddi_wr_obj, 0), "Male": (ddi_mr_obj, 1)}

masked_counts = []
for var in vars_to_clean:
    for sex, (ddi, idx) in ddi_by_sex.items():
        special = codes_matching(ddi, variable_pairs[var][idx], SPECIAL_CODES)
        rows = combined_df["SEX"].eq(sex) & combined_df[var].isin(special)
        combined_df.loc[rows, var] = pd.NA
        masked_counts.append({"variable": var, "sex": sex,
                              "special codes": sorted(special), "set to NaN": int(rows.sum())})

# CHECK: what was masked, and how often
pd.DataFrame(masked_counts)

,variable,sex,special codes,set to NaN
0,EDUCATION_LEVEL,Female,[8],6
1,EDUCATION_LEVEL,Male,[8],4
2,EDUCATION_YEARS,Female,"[96, 97, 98]",61
3,EDUCATION_YEARS,Male,"[96, 97, 98]",59
4,LITERACY_BRIDGE,Female,"[98, 99]",141
5,LITERACY_BRIDGE,Male,"[98, 99]",91
6,LITERACY_PROGRAM_EVER,Female,"[8, 9]",4222
7,LITERACY_PROGRAM_EVER,Male,"[8, 9]",4611
8,WAR_STAYED_HOME,Female,[8],45
9,WAR_STAYED_HOME,Male,[8],28


## 8. Sample restrictions: survey waves and lifelong residents

**Waves.** The extract includes Liberia 2007 and 2013 and Sierra Leone 2008 and 2013. The wave filter keeps these survey years.

**Lifelong residents.** For waves with residence data, people whose residence answer is "Always" are marked as lifelong residents. Residence status is left missing when the survey has no usable residence-years value, as in the 2013 waves; missing status is not treated as non-lifelong.

`YEARS_IN_PLACE` is numeric for observed mover responses. Special codes ("Always", "Visitor", missing) become `NaN`.

In [11]:
allowed_waves = {"Liberia": [2007, 2013], "Sierra Leone": [2008, 2013]}

keep_wave = (
    (combined_df["COUNTRY_NAME"].eq("Liberia") & combined_df["YEAR"].isin(allowed_waves["Liberia"]))
    | (combined_df["COUNTRY_NAME"].eq("Sierra Leone") & combined_df["YEAR"].isin(allowed_waves["Sierra Leone"]))
)
sample = combined_df.loc[keep_wave].copy()

always_codes = {
    "Female": codes_matching(ddi_wr_obj, "RESIDEINTYR", r"\balways\b"),
    "Male": codes_matching(ddi_mr_obj, "RESIDEINTYRMN", r"\balways\b"),
}
print("'Always' codes found:", always_codes)
if not all(always_codes.values()):
    raise ValueError("Could not find the 'Always' residence code in one of the DDIs.")

residence_special = {
    "Female": codes_matching(ddi_wr_obj, "RESIDEINTYR", r"always|visitor|" + SPECIAL_CODES),
    "Male": codes_matching(ddi_mr_obj, "RESIDEINTYRMN", r"always|visitor|" + SPECIAL_CODES),
}
sample["LIFELONG_RESIDENT"] = pd.Series(pd.NA, index=sample.index, dtype="boolean")
for sex, codes in always_codes.items():
    rows = sample["SEX"].eq(sex)
    always_rows = rows & sample["RESIDENCE_YEARS"].isin(codes)
    observed_rows = (
        rows
        & sample["RESIDENCE_YEARS"].notna()
        & ~sample["RESIDENCE_YEARS"].isin(residence_special[sex])
    )
    sample.loc[always_rows, "LIFELONG_RESIDENT"] = True
    sample.loc[observed_rows, "LIFELONG_RESIDENT"] = False

sample["YEARS_IN_PLACE"] = sample["RESIDENCE_YEARS"]
for sex, codes in residence_special.items():
    rows = sample["SEX"].eq(sex) & sample["RESIDENCE_YEARS"].isin(codes)
    sample.loc[rows, "YEARS_IN_PLACE"] = pd.NA

# CHECK
sample.groupby(["COUNTRY_NAME", "YEAR"])["LIFELONG_RESIDENT"].agg(["count", "sum"])

'Always' codes found: {'Female': {95}, 'Male': {94}}


count   sum
COUNTRY_NAME YEAR             
Liberia      2007  12857  6860
             2013      0     0
Sierra Leone 2008  10573  5176
             2013      0     0

## 9. Literacy

Every literacy code gets an explicit category instead of "whole sentence = 1, everything else = 0":

| category | meaning | `LITERACY_STRICT` | `LITERACY_BROAD` |
|---|---|---|---|
| whole | reads a whole sentence | 1 | 1 |
| part | reads part of a sentence | 0 | 1 |
| cannot | cannot read at all | 0 | 0 |
| exclude | blind/visually impaired, or no card in the required language: literacy wasn't measured | NaN | NaN |
| missing | missing, NIU, don't know | NaN | NaN |

`LITERACY_STRICT` is your original definition. `LITERACY_BROAD` matches the DHS standard indicator and works as a robustness check.

**Safety check:** if a code in the data doesn't match any rule, the cell stops with an error listing the code and its label, so a new or unexpected code can never be silently counted as illiterate. If that happens, read the label and add a rule for it.

In [12]:
# Rules are tried in order; the first match wins.
literacy_rules = [
    ("missing", SPECIAL_CODES),
    ("exclude", r"blind|visual"),
    ("exclude", r"card|language"),
    ("whole", r"whole sentence"),
    ("part", r"part"),
    ("cannot", r"cannot|not able|unable"),
]


def classify_codes(ddi, var, rules):
    """{code: category} using the first rule whose pattern matches the code's label."""
    result = {}
    for code, label in labels_for(ddi, var).items():
        for category, pattern in rules:
            if re.search(pattern, label, flags=re.IGNORECASE):
                result[code] = category
                break
    return result


literacy_categories = {
    "Female": classify_codes(ddi_wr_obj, "LIT2", literacy_rules),
    "Male": classify_codes(ddi_mr_obj, "LIT2MN", literacy_rules),
}

sample["LIT_CATEGORY"] = pd.Series(pd.NA, index=sample.index, dtype="string")
for sex, mapping in literacy_categories.items():
    rows = sample["SEX"].eq(sex)
    sample.loc[rows, "LIT_CATEGORY"] = sample.loc[rows, "LITERACY"].map(mapping)

# SAFETY CHECK: stop if any observed code wasn't classified
unclassified = sample.loc[sample["LITERACY"].notna() & sample["LIT_CATEGORY"].isna(), ["SEX", "LITERACY"]]
if len(unclassified):
    details = unclassified.value_counts().reset_index()
    details["label"] = [labels_for(ddi_by_sex[s][0], variable_pairs["LITERACY"][ddi_by_sex[s][1]]).get(c)
                        for s, c in zip(details["SEX"], details["LITERACY"])]
    raise ValueError(f"Unclassified literacy codes; add a rule for them:\n{details}")

sample["LITERACY_STRICT"] = sample["LIT_CATEGORY"].map({"whole": 1, "part": 0, "cannot": 0}).astype("Float64")
sample["LITERACY_BROAD"] = sample["LIT_CATEGORY"].map({"whole": 1, "part": 1, "cannot": 0}).astype("Float64")

# CHECK: how each code was classified, with counts
check = sample.groupby(["SEX", "LITERACY", "LIT_CATEGORY"], dropna=False).size().rename("n").reset_index()
check["label"] = [labels_for(ddi_by_sex[s][0], variable_pairs["LITERACY"][ddi_by_sex[s][1]]).get(c)
                  for s, c in zip(check["SEX"], check["LITERACY"])]
check

,SEX,LITERACY,LIT_CATEGORY,n,label
0,Female,0,cannot,25520,Cannot read at all
1,Female,11,part,2660,Able to read only part of sentence
2,Female,12,whole,11981,Able to read whole sentence
3,Female,21,exclude,43,No card with required language
4,Female,22,exclude,18,Blind or visually impaired
5,Female,98,missing,141,Missing
6,Male,0,cannot,8220,Cannot read at all
7,Male,11,part,1696,Able to read only parts of sentence
8,Male,12,whole,10611,Able to read whole sentence
9,Male,21,exclude,38,No card with required language


## 10. Build the analysis sample and check it

The analysis sample keeps lifelong residents with a measured literacy answer. Waves without usable residence-years data have unknown lifelong-resident status and are excluded from this strict sample.

The cell writes two country-level analysis files and four wave-specific files. Wave files retain all records in their country-year; the analysis files apply the lifelong-resident and measured-literacy restrictions. All six use the same harmonized columns and include `GENDER`.

The attrition table shows *why* records were dropped. The education-by-literacy table checks whether respondents with secondary or higher education were classified as literate.

In [13]:
has_literacy = sample["LITERACY_STRICT"].notna()
lifelong = sample["LIFELONG_RESIDENT"]
sample["GENDER"] = sample["SEX"]

attrition = pd.DataFrame({
    "Country": sample["COUNTRY_NAME"],
    "Wave-filtered records": 1,
    "Not lifelong resident only": lifelong.eq(False).fillna(False) & has_literacy,
    "No literacy measure only": lifelong.eq(True).fillna(False) & ~has_literacy,
    "Fail both filters": lifelong.eq(False).fillna(False) & ~has_literacy,
    "Residence status unavailable": lifelong.isna(),
    "Retained": lifelong.eq(True).fillna(False) & has_literacy,
}).groupby("Country").sum()
display(attrition)

retained = lifelong.fillna(False) & has_literacy
postwar_clean_df = sample.loc[retained].copy()
liberia_clean_df = postwar_clean_df.loc[postwar_clean_df["COUNTRY_NAME"].eq("Liberia")].copy()
sierra_leone_clean_df = postwar_clean_df.loc[postwar_clean_df["COUNTRY_NAME"].eq("Sierra Leone")].copy()

country_output_columns = [
    "COUNTRY_NAME", "YEAR", "GENDER", "ADMIN_AREA", "ADMIN_AREA_NAME",
    "ADMIN_UNIT_NAME", "ADMIN_UNIT_LEVEL", "GPS_LATITUDE", "GPS_LONGITUDE",
    *variable_pairs.keys(), "LIFELONG_RESIDENT", "YEARS_IN_PLACE",
    "LIT_CATEGORY", "LITERACY_STRICT", "LITERACY_BROAD",
]

analysis_outputs = {
    "liberia_analysis_clean.csv": liberia_clean_df,
    "sierra_leone_analysis_clean.csv": sierra_leone_clean_df,
}
for filename, country_df in analysis_outputs.items():
    output_path = os.path.join(dhs_processed_dir, filename)
    country_df[country_output_columns].to_csv(output_path, index=False)
    exported = pd.read_csv(output_path)
    assert list(exported.columns) == country_output_columns
    assert exported["GENDER"].notna().all()
    assert set(exported["GENDER"]) == {"Female", "Male"}
    assert exported["GPS_LATITUDE"].notna().equals(exported["GPS_LONGITUDE"].notna())
    print(f"{filename}: {exported.shape}, gender counts: {exported['GENDER'].value_counts().to_dict()}")

wave_outputs = {
    "liberia_2007_clean.csv": ("Liberia", 2007),
    "liberia_2013_clean.csv": ("Liberia", 2013),
    "sierra_leone_2008_clean.csv": ("Sierra Leone", 2008),
    "sierra_leone_2013_clean.csv": ("Sierra Leone", 2013),
}
for filename, (country, year) in wave_outputs.items():
    wave_df = sample.loc[
        sample["COUNTRY_NAME"].eq(country) & sample["YEAR"].eq(year),
        country_output_columns,
    ]
    if wave_df.empty:
        raise ValueError(f"No records found for {country} {year}.")
    output_path = os.path.join(dhs_processed_dir, filename)
    wave_df.to_csv(output_path, index=False)
    exported = pd.read_csv(output_path)
    assert list(exported.columns) == country_output_columns
    assert exported["COUNTRY_NAME"].eq(country).all()
    assert exported["YEAR"].eq(year).all()
    assert exported["GENDER"].notna().all()
    assert set(exported["GENDER"]) == {"Female", "Male"}
    assert exported["GPS_LATITUDE"].notna().equals(exported["GPS_LONGITUDE"].notna())
    if year == 2013:
        assert exported["LIFELONG_RESIDENT"].isna().all()
        assert exported["ADMIN_UNIT_LEVEL"].isin(["County", "District"]).all()
    else:
        assert exported["ADMIN_UNIT_LEVEL"].isin(["County group", "District group"]).all()
    print(f"{filename}: {exported.shape}, gender counts: {exported['GENDER'].value_counts().to_dict()}")

# CHECK: education level vs. literacy category
edu_labels = {**labels_for(ddi_mr_obj, "EDUCLVLMN"), **labels_for(ddi_wr_obj, "EDUCLVL")}
pd.crosstab(postwar_clean_df["EDUCATION_LEVEL"].map(edu_labels), postwar_clean_df["LIT_CATEGORY"])

,Wave-filtered records,Not lifelong resident only,No literacy measure only,Fail both filters,Residence status unavailable,Retained
Country,,,,,,
Liberia,26458,5923,46,74,13601,6814
Sierra Leone,34574,5357,55,40,24001,5121


liberia_analysis_clean.csv: (6814, 30), gender counts: {'Female': 3865, 'Male': 2949}
sierra_leone_analysis_clean.csv: (5121, 30), gender counts: {'Female': 3308, 'Male': 1813}
liberia_2007_clean.csv: (13101, 30), gender counts: {'Female': 7092, 'Male': 6009}
liberia_2013_clean.csv: (13357, 30), gender counts: {'Female': 9239, 'Male': 4118}
sierra_leone_2008_clean.csv: (10654, 30), gender counts: {'Female': 7374, 'Male': 3280}
sierra_leone_2013_clean.csv: (23920, 30), gender counts: {'Female': 16658, 'Male': 7262}


LIT_CATEGORY,cannot,part,whole
EDUCATION_LEVEL,,,
Higher,0,0,272
No education,5594,15,5
Primary,1523,1040,543
Secondary,0,0,2942


## 11. Weighted literacy rates

DHS samples aren't self-weighting, so rates use the person weight. A weighted share is `sum(weight × literate) / sum(weight)`, computed per group. This version is vectorized: no loop, one `groupby`.

Compare strict and broad side by side: a big gap means your choice of definition matters, which is worth reporting.

In [14]:
def weighted_share(df, col, by):
    """Weighted mean of a 0/1 column within each group."""
    d = df.loc[df[col].notna() & df["WEIGHT"].notna()]
    sums = d.assign(weighted=d[col] * d["WEIGHT"]).groupby(by)[["weighted", "WEIGHT"]].sum()
    return sums["weighted"] / sums["WEIGHT"]


groups = ["COUNTRY_NAME", "YEAR", "SEX"]
literacy_rates = pd.DataFrame({
    "strict (whole sentence)": weighted_share(postwar_clean_df, "LITERACY_STRICT", groups),
    "broad (part or whole)": weighted_share(postwar_clean_df, "LITERACY_BROAD", groups),
    "unweighted strict": postwar_clean_df.groupby(groups)["LITERACY_STRICT"].mean(),
    "n": postwar_clean_df.groupby(groups).size(),
}).round(3)
literacy_rates

strict (whole sentence)  broad (part or whole)  \
COUNTRY_NAME YEAR SEX                                                      
Liberia      2007 Female                    0.277                  0.395   
                  Male                      0.542                  0.669   
Sierra Leone 2008 Female                    0.154                    0.2   
                  Male                      0.281                  0.338   

                          unweighted strict     n  
COUNTRY_NAME YEAR SEX                              
Liberia      2007 Female              0.257  3865  
                  Male                0.547  2949  
Sierra Leone 2008 Female               0.18  3308  
                  Male                0.308  1813

# Uppsala Conflict Data Program

**Note for the merge:** UCDP's `adm_1` is the county in Liberia, and `adm_2` the district in Sierra Leone. The DHS regions from step 6 are *groups* of these, so the next step is a lookup table assigning each county/district to its DHS region, then summing events and deaths by region.

In [15]:
ucpd_path = os.path.join(ucdp_raw_dir, "ucdp_ged_global.csv")

# low_memory=False reads the file in one pass, which fixes the mixed-type warning
ucpd = pd.read_csv(ucpd_path, low_memory=False)

In [16]:
violence_labels = {1: "State-based", 2: "Non-state", 3: "One-sided"}

ged = ucpd[ucpd["country"].isin(["Liberia", "Sierra Leone"])].copy()
ged["violence_type"] = ged["type_of_violence"].map(violence_labels)

# Liberia: adm_1 = county. Sierra Leone: adm_1 = province, adm_2 = district.
ged["admin_unit"] = ged["adm_1"].where(ged["country"].eq("Liberia"), ged["adm_2"])
ged["admin_unit"] = ged["admin_unit"].fillna("Unknown / not geocoded")

# CHECK: the units you'll need to map onto DHS regions
ged.groupby("country")["admin_unit"].value_counts()

country       admin_unit                 
Liberia       Montserrado county             115
              Lofa county                     94
              Nimba county                    50
              Bong county                     42
              Bomi county                     40
              Grand Cape Mount county         39
              River Cess county               31
              Grand Bassa county              28
              Grand Gedeh county              28
              Sinoe county                    22
              Unknown / not geocoded          21
              Maryland county                 13
              Margibi county                  11
              Grand Kru county                 9
              River Gee county                 5
              Gbarpolu county                  3
Sierra Leone  Kono district                  221
              Kenema district                215
              Port Loko district             157
              Kailahun dist

In [17]:
country_summary = (
    ged.groupby(["country", "violence_type"])
    .agg(
        events=("id", "count"),
        deaths_best=("best", "sum"),
        deaths_low=("low", "sum"),
        deaths_high=("high", "sum"),
        first_year=("year", "min"),
        last_year=("year", "max"),
    )
    .reset_index()
)
country_summary

,country,violence_type,events,deaths_best,deaths_low,deaths_high,first_year,last_year
0,Liberia,Non-state,60,3026,1372,5703,1990,1996
1,Liberia,One-sided,371,17160,17160,18874,1989,2024
2,Liberia,State-based,120,3048,2947,3267,1989,2003
3,Sierra Leone,Non-state,12,81,81,81,1991,1996
4,Sierra Leone,One-sided,852,8893,8892,14021,1991,2021
5,Sierra Leone,State-based,633,11470,11466,18425,1991,2001


## Conflict-event animation

Unchanged from your version. It is slow to render, so you may want to skip it while working on the cleaning. (The `KeyboardInterrupt` you saw earlier came from running it in debug mode with a breakpoint, not from a bug.)

In [ ]:
RUN_ANIMATION = False
if RUN_ANIMATION:
    # Assumes the same imports and `ucpd` DataFrame as the original notebook
    # (pandas as pd, numpy as np, geopandas as gpd, matplotlib.pyplot as plt,
    #  matplotlib.animation as animation, to_rgba, Line2D, requests, os, Video, Image).

    data_dir = os.path.join(project_dir, "data")
    start_date, end_date = "1989-12-01", "2003-08-31"
    west, east, south, north = -14.0, -7.0, 4.0, 10.6
    region = ["Liberia", "Sierra Leone", "Guinea", "Ivory Coast"]
    n_factions = 9
    step = 1
    fps = 12

    fig_w, fig_h = 10, 11.2
    video_dpi = 150
    gif_dpi = 90
    display_width = 900

    ev = ucpd.copy()
    ev["date"] = pd.to_datetime(ev["date_start"], errors="coerce")
    ev = ev[
        ev["date"].between(start_date, end_date)
        & ev["country"].isin(region)
        & ev["longitude"].between(west, east)
        & ev["latitude"].between(south, north)
    ].sort_values(["date", "id"]).reset_index(drop=True)

    anchor_candidates = combined_df.loc[
        combined_df["COUNTRY_NAME"].isin(["Liberia", "Sierra Leone"]),
        ["COUNTRY_NAME", "DHSID", "GPS_LATITUDE", "GPS_LONGITUDE"],
    ].dropna().drop_duplicates(subset=["COUNTRY_NAME", "DHSID"])
    anchor_points = (
        anchor_candidates.groupby("COUNTRY_NAME", group_keys=False)
        .sample(n=1, random_state=4538)
        .set_index("COUNTRY_NAME")
    )
    if set(anchor_points.index) != {"Liberia", "Sierra Leone"}:
        raise ValueError("Could not select one GPS cluster in both Liberia and Sierra Leone.")

    def faction(row):
        """Armed group behind the event: the first side that is not a government or civilians."""
        for side in (row["side_a"], row["side_b"]):
            side_name = str(side)
            if not side_name.startswith("Government of") and side_name != "Civilians":
                return side_name
        return str(row["side_a"])

    ev["faction"] = ev.apply(faction, axis=1)
    top = ev["faction"].value_counts().head(n_factions).index.tolist()
    palette = ["#c0392b", "#1f618d", "#e08e0b", "#1e8449", "#7d3c98",
               "#17a2b8", "#d35487", "#7b5a3a", "#b7950b", "#34495e"]
    fcolors = {f: palette[i % len(palette)] for i, f in enumerate(top)}

    n = len(ev)
    xy = ev[["longitude", "latitude"]].to_numpy()
    rgba = np.array([to_rgba(fcolors.get(f, "#555555"), 0.65) for f in ev["faction"]])
    sizes = (10 + 3.5 * np.sqrt(ev["best"].clip(lower=0))).to_numpy()

    months = pd.period_range(ev["date"].min(), ev["date"].max(), freq="M")
    ev["mi"] = (ev["date"].dt.year - months[0].year) * 12 + ev["date"].dt.month - months[0].month
    monthly = ev.groupby("mi").size().reindex(range(len(months)), fill_value=0)

    def boundaries(iso, level):
        """geoBoundaries admin units, downloaded once and cached in data_dir."""
        path = os.path.join(geography_dir, f"geoboundaries_{iso.lower()}_{level.lower()}.geojson")
        if not os.path.exists(path):
            try:
                meta = requests.get(
                    f"https://www.geoboundaries.org/api/current/gbOpen/{iso}/{level}/", timeout=60
                ).json()
                url = meta["gjDownloadURL"]
            except Exception:
                url = ("https://github.com/wmgeolab/geoBoundaries/raw/main/releaseData/"
                       f"gbOpen/{iso}/{level}/geoBoundaries-{iso}-{level}.geojson")
            response = requests.get(url, timeout=120)
            response.raise_for_status()
            with open(path, "wb") as f:
                f.write(response.content)
        return gpd.read_file(path)

    world = gpd.read_file(os.path.join(geography_dir, "natural_earth_admin0.zip"))
    focus = world[world["ADMIN"].isin(["Liberia", "Sierra Leone"])]
    admin = pd.concat([boundaries("LBR", "ADM1"), boundaries("SLE", "ADM2")])

    fig, (ax, tl) = plt.subplots(
        2, 1, figsize=(fig_w, fig_h), gridspec_kw={"height_ratios": [8, 1], "hspace": 0.06}
    )
    fig.subplots_adjust(left=0.03, right=0.97, top=0.93, bottom=0.05)

    ax.set_facecolor("#dbe9f1")
    world.plot(ax=ax, color="#f3f0e8", edgecolor="#555555", linewidth=1.0)
    focus.plot(ax=ax, color="#fbfaf6", edgecolor="none")
    admin.boundary.plot(ax=ax, color="#a9a9a9", linewidth=0.5, zorder=2)
    focus.boundary.plot(ax=ax, color="black", linewidth=1.6, zorder=2.5)
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_xticks([])
    ax.set_yticks([])

    radius_styles = {
        2: {"color": "#d1495b", "linestyle": "-"},
        5: {"color": "#edae49", "linestyle": "--"},
        10: {"color": "#00798c", "linestyle": ":"},
    }
    country_colors = {"Liberia": "#29335c", "Sierra Leone": "#6a994e"}
    country_markers = {"Liberia": "*", "Sierra Leone": "D"}
    for country, anchor in anchor_points.iterrows():
        anchor_geo = gpd.GeoDataFrame(
            {"COUNTRY_NAME": [country]},
            geometry=gpd.points_from_xy(
                [anchor["GPS_LONGITUDE"]], [anchor["GPS_LATITUDE"]]
            ),
            crs="EPSG:4326",
        )
        anchor_utm = anchor_geo.to_crs(anchor_geo.estimate_utm_crs())
        for radius_km, style in radius_styles.items():
            bubble = anchor_utm.copy()
            bubble.geometry = anchor_utm.geometry.buffer(radius_km * 1000)
            bubble.to_crs("EPSG:4326").boundary.plot(
                ax=ax, linewidth=1.8, zorder=2.8, **style
            )
        ax.scatter(
            anchor["GPS_LONGITUDE"],
            anchor["GPS_LATITUDE"],
            marker=country_markers[country],
            s=105,
            color=country_colors[country],
            edgecolors="white",
            linewidths=0.8,
            zorder=4,
        )

    print("Randomly selected DHS cluster centers (fixed seed):")
    print(anchor_points[["DHSID", "GPS_LATITUDE", "GPS_LONGITUDE"]])

    for name, x, y in [("GUINEA", -10.3, 10.15), ("CÔTE\nD'IVOIRE", -7.55, 7.2)]:
        ax.text(x, y, name, ha="center", va="center", fontsize=11,
                color="#8a8a8a", fontweight="bold", zorder=1)
    for name, x, y in [("Freetown", -13.23, 8.48), ("Monrovia", -10.80, 6.30)]:
        ax.plot(x, y, marker="s", ms=5, color="black", zorder=6)
        ax.annotate(name, (x, y), xytext=(5, -11), textcoords="offset points", fontsize=9, zorder=6)

    dots = ax.scatter([], [], s=[], edgecolors="white", linewidths=0.3, zorder=3)
    ring = ax.scatter([], [], s=[], facecolors="none", edgecolors="black",
                      linewidths=1.5, zorder=5)
    title = ax.set_title("", fontsize=12, loc="left")

    faction_handles = [
        Line2D([], [], marker="o", ls="", color=color, ms=8, label=name)
        for name, color in fcolors.items()
    ]
    faction_handles += [
        Line2D([], [], marker="o", ls="", color="#555555", ms=8, label="Other actors"),
        Line2D([], [], marker="o", ls="", mfc="none", mec="black", mew=1.5,
               ms=10, label="Current event"),
    ]
    faction_legend = ax.legend(
        handles=faction_handles, loc="lower left", frameon=True, framealpha=0.92,
        fontsize=8.5, title="Faction", title_fontsize=9.5,
    )
    ax.add_artist(faction_legend)
    bubble_handles = [
        Line2D([], [], color=style["color"], linestyle=style["linestyle"],
               linewidth=1.8, label=f"{radius_km} km radius")
        for radius_km, style in radius_styles.items()
    ]
    bubble_handles += [
        Line2D([], [], marker=country_markers[country], ls="", color=country_colors[country],
               markeredgecolor="white", ms=9, label=f"{country} random cluster")
        for country in ["Liberia", "Sierra Leone"]
    ]
    ax.legend(
        handles=bubble_handles, loc="upper right", frameon=True, framealpha=0.92,
        fontsize=8.5, title="Distance bubbles", title_fontsize=9.5,
    )

    tl.bar(range(len(months)), monthly.values, color="#bbbbbb", width=1.0)
    marker = tl.axvline(0, color="black", linewidth=1.5)
    jan = [i for i, month in enumerate(months) if month.month == 1]
    tl.set_xticks(jan)
    tl.set_xticklabels([months[i].year for i in jan], fontsize=9, rotation=45)
    tl.set_xlim(-0.5, len(months) - 0.5)
    tl.set_ylabel("Events", fontsize=9)
    tl.tick_params(axis="y", labelsize=8)
    for spine in ["top", "right"]:
        tl.spines[spine].set_visible(False)

    n_steps = int(np.ceil(n / step))
    hold = fps * 3

    def update(k):
        i = min((k + 1) * step, n)
        j = max(i - step, 0)
        dots.set_offsets(xy[:i])
        dots.set_sizes(sizes[:i])
        dots.set_facecolors(rgba[:i])
        ring.set_offsets(xy[j:i])
        ring.set_sizes(sizes[j:i] * 3 + 50)
        row = ev.iloc[i - 1]
        marker.set_xdata([row["mi"], row["mi"]])
        title.set_text(
            f"{row['date'].strftime('%d %B %Y')}   ·   event {i} of {n}\n"
            f"{row['side_a']} vs {row['side_b']}   ·   "
            f"{row.get('where_coordinates', '')}, {row['country']}   ·   {int(row['best'])} deaths"
        )
        return dots, ring, marker, title

    anim = animation.FuncAnimation(fig, update, frames=n_steps + hold, interval=1000 / fps)

    out = os.path.join(output_dir, "conflict_events_large")
    if animation.writers.is_available("ffmpeg"):
        anim.save(out + ".mp4", writer="ffmpeg", fps=fps, dpi=video_dpi,
                  extra_args=["-pix_fmt", "yuv420p"])
        shown = Video(out + ".mp4", embed=True, html_attributes="controls loop", width=display_width)
    else:
        anim.save(out + ".gif", writer="pillow", fps=fps, dpi=gif_dpi)
        shown = Image(filename=out + ".gif", width=display_width)
    plt.close(fig)
    print(f"{n} events, {ev['date'].min():%d %b %Y} to {ev['date'].max():%d %b %Y}")
    shown

Randomly selected DHS cluster centers (fixed seed):
                       DHSID  GPS_LATITUDE  GPS_LONGITUDE
COUNTRY_NAME                                             
Liberia       LB200700000042      7.677866      -10.67988
Sierra Leone  SL200800000266       8.83719      -12.59338
2133 events, 01 Dec 1989 to 08 Aug 2003
